# rl17_train

> Phase 17: `trainRL17`, `trainRL`'s GRPO loop (its step pieces in `rl17_update`) with three changes: each step's boards are split across team sizes (`mixN`) with scripted liars on a share of them (`liarShare`); the reference and old log-probabilities are computed once per step and reused by `epochs` update passes with a clipped reply-level ratio (`clip`); replies can be sampled under the reply schema (`guided`), and `lrFixed` switches the per-step controller off. At the defaults one step equals `trainRL`'s (tested on the tiny model). Training needs the `[train]` extra; the offline tests do not.

In [ ]:
#| default_exp rl17_train

In [ ]:
#| hide
from nbdev.showdoc import *

## The training loop

In [ ]:
#| export
import json
import time
from pathlib import Path

import numpy as np

from explRL.clue_agents import ClueLiar
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_roll import coopRows, rollout
from explRL.clue_sft import libVersions, loadModel, loraConfig
from explRL.llm_runs import LLMConfig
from explRL.local_policy import LocalPolicy, pickDevice
from explRL.rl_adv import groupAdvantages, stepSamples, trainingReturns
from explRL.rl_control import haltReason, invalidShare, jumpGuard, microBatches, nanMean, nextLr, progressBar, stepLine, warmLr
from explRL.rl_torch import freeGPU, loadCheckpoint, probeKL, probeLogps, restore, saveCheckpoint, saveMemory, snapshot
from explRL.rl17_grid import configHash17, runFields17
from explRL.rl17_update import boardGroups, ppoUpdate, saveTurns


def makePolicies17(cfg, model, tok, dev):
    "As rl_torch.makePolicies (one shared adapter), with constrained replies when cfg.guided."
    vllm = dev == 'cuda' and LocalPolicy(cfg.model, device=dev, engine='auto', tokenizer=tok).engine == 'vllm'
    pol = LocalPolicy(cfg.model, device=dev, maxNew=cfg.maxNew, temperature=cfg.temperature, seed=cfg.trainSeed,
                      engine='vllm' if vllm else 'hf', hf=None if vllm else model, tokenizer=tok, adapterName='default',
                      guided=cfg.guided)
    return pol, vllm


def trainRL17(cfg, settings=CLUE_SETTINGS, show=True):
    "trainRL with mixed team sizes and liars, clipped passes, constrained replies and a fixed rate (rl17_grid.RL17Config)."
    import torch
    from peft import get_peft_model
    assert cfg.advantage == 'board' and not cfg.perRole, 'Phase 17 uses board-level advantages and one shared adapter'
    out, dev = Path(cfg.out), pickDevice(cfg.device)
    out.mkdir(parents=True, exist_ok=True)
    (out / 'config.json').write_text(json.dumps(dict(runFields17(cfg), hash=configHash17(cfg), versions=libVersions()), indent=1) + '\n')
    torch.manual_seed(cfg.trainSeed)
    tok, base = loadModel(cfg.model, dev)
    model = get_peft_model(base, loraConfig(cfg.loraR, cfg.loraAlpha), adapter_name='default')
    saveMemory(model, cfg.gradCkpt)
    params = [p for n, p in model.named_parameters() if 'lora_' in n]
    for p in params:
        p.requires_grad_(True)
    rate = cfg.lrFixed if cfg.lrFixed is not None else cfg.lr
    opt = torch.optim.AdamW(params, lr=rate)
    start = loadCheckpoint(model, opt, out, configHash17(cfg), ['default'])
    if (out / 'halted.json').exists():
        print('halted:', json.loads((out / 'halted.json').read_text())['reason'])
        return str(out / 'final')
    mp = out / 'metrics.jsonl'
    history = [h for h in map(json.loads, open(mp)) if h['step'] <= start] if mp.exists() else []
    mp.write_text(''.join(json.dumps(h) + '\n' for h in history))
    lr = history[-1].get('lrNext', rate) if history else rate
    pol, vllm = makePolicies17(cfg, model, tok, dev)
    lcfg = lambda N, nl: LLMConfig(task='clue', channel=cfg.channel, N=N, nLiar=nl, turns=cfg.turns, model=cfg.model,
                                   world={**{k: settings[k] for k in BOARD}, 'scheme': cfg.scheme})
    ctl = cfg if cfg.lrFixed is None else type(cfg)(**{**cfg.__dict__, 'klTarget': 0.0})
    bar = progressBar(cfg.steps, start, f'{cfg.scheme} RL17', show)
    kept, rolls = None, 0
    with open(mp, 'a') as log:
        for s in range(start, cfg.steps):
            t0 = time.time()

            def play():
                model.eval()
                if vllm:
                    model.save_pretrained(out / 'live')
                pol.calls = s * 100_000
                if vllm:
                    pol.setAdapter(str(out / 'live'))
                return [(lcfg(N, nl), rollout(lcfg(N, nl), seeds, pol, {i: ClueLiar for i in range(nl)} if nl else None,
                                                       keepPrompts=True))
                        for N, nl, seeds in boardGroups(cfg, s)]

            rowsOf = lambda gs: [r for c, ro in gs for r in coopRows(c, ro)]
            jumped = lambda gs: kept is not None and invalidShare(rowsOf(gs)) > cfg.haltJump
            gs, back, rolls = jumpGuard(play, jumped, lambda: restore(params, opt, kept), rolls)
            lr *= 0.5 ** back
            if gs is None:
                reason = f'jump: invalid replies above {cfg.haltJump:g} after two roll-backs in a row (step {s + 1})'
                (out / 'halted.json').write_text(json.dumps(dict(step=s, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')
                break
            Rs, signal, samples, rng = [], 0, [], np.random.default_rng([cfg.trainSeed, s])
            for c, ro in gs:
                R = trainingReturns(ro, cfg.formatPenalty, cfg.wrongPenalty)
                adv = groupAdvantages(R.reshape(-1, cfg.group, c.N)).reshape(R.shape)
                Rs.append(R[:, c.nLiar:])
                signal += len(stepSamples(ro, adv))
                samples += stepSamples(ro, adv, cfg.turnFraction, rng)
            t1 = time.time()
            model.train()
            for g in opt.param_groups:
                g['lr'] = warmLr(lr, s, cfg.warmup)
            probe = microBatches(samples[::max(1, len(samples) // cfg.probe)][:cfg.probe], cfg)
            before = probeLogps(model, tok, probe, ['default'], False, cfg.maxLen, dev)
            snap = snapshot(params, opt) if ctl.klTarget or cfg.haltJump is not None else None
            stats = ppoUpdate(model, tok, opt, params, samples, microBatches(samples, cfg), cfg.kl, cfg.maxLen, dev,
                              cfg.epochs, cfg.clip, seed=cfg.trainSeed * 100_003 + s)
            stepKL = probeKL(before, probeLogps(model, tok, probe, ['default'], False, cfg.maxLen, dev))
            lrNext, rejected = nextLr(lr, stepKL, ctl, warming=s + 1 < cfg.warmup, trained=bool(samples))
            if rejected:
                restore(params, opt, snap)
            kept = snap if cfg.haltJump is not None else None
            rows = rowsOf(gs)
            m = dict(step=s + 1, rolloutSeconds=round(t1 - t0, 2), updateSeconds=round(time.time() - t1, 2),
                     meanReturn=float(np.mean(np.concatenate([r.ravel() for r in Rs]))),
                     welfare=float(np.mean([r['welfare'] for r in rows])), sharePerHeld=float(np.mean([r['sharePerHeld'] for r in rows])),
                     invalid=invalidShare(rows), soloShare=nanMean([r['soloShare'] for r in rows]), signalTurns=signal,
                     samples=len(samples), lr=warmLr(lr, s, cfg.warmup), stepKL=stepKL, rejected=rejected, lrNext=lrNext,
                     gradCkpt=bool(getattr(model, 'is_gradient_checkpointing', False)), **stats,
                     **({} if cfg.haltJump is None else dict(rolledBack=back)))
            log.write(json.dumps(m) + '\n')
            log.flush()
            history.append(m)
            bar.write(stepLine(m, cfg.steps)) if show else None
            bar.update(1)
            lr = lrNext
            reason = haltReason(history, cfg)
            if (s + 1) % cfg.ckptEvery == 0 or s + 1 == cfg.steps or reason:
                saveCheckpoint(model, opt, s + 1, out, configHash17(cfg), None)
                saveTurns(out, s + 1, gs)
            if reason:
                (out / 'halted.json').write_text(json.dumps(dict(step=s + 1, reason=reason)) + '\n')
                bar.write(f'halted: {reason}')
                break
    bar.close()
    model.save_pretrained(out / 'final')
    del model, base, opt, params
    freeGPU(pol)
    return str(out / 'final')

## Tests

In [ ]:
import numpy as np
from explRL.rl17_train import *
from explRL.rl17_grid import RL17Config

In [ ]:
#| train
# With the [train] extra, no download: at the defaults trainRL17 equals trainRL (metrics and final adapter) on the tiny model;
# a mixed step with liars and clipped passes runs; its checkpoint carries configHash17
import json
import tempfile
import torch
from dataclasses import asdict
from explRL.clue_rl import RLConfig, trainRL
import explRL.rl17_train as T
from explRL.tiny_model import tinyModel
from safetensors.torch import load_file
_tr = T.trainingReturns
with tempfile.TemporaryDirectory() as d:
    tiny = tinyModel(d + '/tiny')
    kw = dict(steps=2, seedsPerStep=2, group=3, turns=3, ckptEvery=1, maxNew=24, lr=1e-3, haltRun=99)
    import explRL.clue_rl as C
    for mod in (C, T):                       # the tiny model's replies are invalid: the same random returns stand in for both
        rng = np.random.default_rng(0)
        mod.trainingReturns = (lambda r: (lambda ro, *a: _tr(ro, *a) + r.normal(size=np.shape(ro.returns))))(rng)
        if mod is C:
            trainRL(RLConfig(tiny, f'{d}/a', **kw))
        else:
            trainRL17(RL17Config(tiny, f'{d}/b', **kw))
    strip = lambda p: [{k: v for k, v in json.loads(x).items() if 'Seconds' not in k} for x in open(p)]
    assert strip(f'{d}/a/metrics.jsonl') == strip(f'{d}/b/metrics.jsonl')
    wa, wb = load_file(f'{d}/a/final/adapter_model.safetensors'), load_file(f'{d}/b/final/adapter_model.safetensors')
    assert wa.keys() == wb.keys() and all(torch.equal(wa[k], wb[k]) for k in wa)
    T.trainingReturns = (lambda r: (lambda ro, *a: _tr(ro, *a) + r.normal(size=np.shape(ro.returns))))(np.random.default_rng(1))
    cm = RL17Config(tiny, f'{d}/m', **{**kw, 'seedsPerStep': 4}, mixN=((4, 0.5), (8, 0.5)), liarShare=0.5, epochs=2, clip=0.2, kl=0.0, lrFixed=1e-3)
    trainRL17(cm)
    m = [json.loads(x) for x in open(f'{d}/m/metrics.jsonl')]
    assert [x['step'] for x in m] == [1, 2] and all(x['samples'] > 0 and np.isfinite(x['loss']) for x in m)
    assert json.load(open(f'{d}/m/config.json'))['hash'] == T.configHash17(cm)
    T.trainingReturns = _tr
    C.trainingReturns = _tr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()